# CoffeeKing | EDA, Data Cleaning, & Analysis 
### SQL for Data Science Capstone | UC Davis
**Dataset:** Yelp Academic Dataset - Nashville, TN

### Step 0: List Available Tables
This command lists every table that exists in the current schema. It confirms the Yelp dataset loaded correctly and shows the five tables we will work with: business, review, tip, checkin, and user.


In [0]:
SHOW TABLES;

database,tableName,isTemporary
default,business,false
default,checkin,false
default,review,false
default,tip,false
default,user,false


### Step 1: Row Counts Across All Tables
This query stacks a count from each of the five tables using UNION ALL, so we get one combined result showing how many records live in business, review, tip, checkin, and user. The goal is to understand the overall size of the dataset before doing any cleaning or filtering.

In [0]:
-- =============================================
-- STEP 1: Row count for all 5 tables
-- =============================================
SELECT COUNT(*) AS row_count FROM business
UNION ALL
SELECT COUNT(*) FROM review
UNION ALL
SELECT COUNT(*) FROM tip
UNION ALL
SELECT COUNT(*) FROM checkin
UNION ALL
SELECT COUNT(*) FROM user;

row_count
6971
919271
54115
6178
180431


### Step 2: Explore the Business Table Structure
This shows every column in the business table along with its data type. Reviewing the schema first helps us know which fields are usable for analysis, such as stars, categories, postal_code, review_count, and is_open.

In [0]:
-- =============================================
-- STEP 2: Explore columns in business table
-- =============================================
DESCRIBE TABLE business;

col_name,data_type,comment
address,string,null
attributes,string,null
business_id,string,null
categories,string,null
city,string,null
hours,string,null
is_open,bigint,null
latitude,double,null
longitude,double,null
name,string,null


### Step 3: Check for Missing Values
This counts how many non null values exist in each key column of the business table and compares each count to the total row count. If a column's count matches total_rows, that column has no missing data. This is a core data cleaning step before trusting any column for analysis.

In [0]:
-- =============================================
-- STEP 3: Check NULLs in business table
-- =============================================
SELECT
  COUNT(*)                          AS total_rows,
  COUNT(business_id)                AS has_business_id,
  COUNT(name)                       AS has_name,
  COUNT(address)                    AS has_address,
  COUNT(stars)                      AS has_stars,
  COUNT(categories)                 AS has_categories,
  COUNT(hours)                      AS has_hours,
  COUNT(postal_code)                AS has_postal_code,
  COUNT(is_open)                    AS has_is_open,
  COUNT(review_count)               AS has_review_count
 FROM business;

total_rows,has_business_id,has_name,has_address,has_stars,has_categories,has_hours,has_postal_code,has_is_open,has_review_count
6971,6971,6971,6686,6971,6971,6971,6967,6971,6971


### Step 4: Validate the Stars Column
This groups every business by its star rating and counts how many businesses fall into each value. The purpose is to confirm the stars column only contains valid Yelp values and has no typos, negative numbers, or out of range entries.



In [0]:
-- =============================================
-- STEP 4: Validate star ratings in business
-- =============================================
SELECT
  stars,
  COUNT(*) AS count
FROM business
GROUP BY stars
ORDER BY stars;

stars,count
1.0,99
1.5,253
2.0,420
2.5,573
3.0,807
3.5,1161
4.0,1493
4.5,1347
5.0,818


**Result:** Only 9 distinct values appear, all valid Yelp ratings from 1.0 to 5.0 in 0.5 increments. The stars column is clean with zero dirty data. Most businesses cluster at 4.0 stars (1,493 total), which points to positivity bias, a common pattern where customers are more likely to leave a review when they had a good experience.

Verdict: Only 9 distinct values, all valid Yelp values (1.0 to 5.0 in 0.5 steps). Zero dirty data here. Stars column is clean and ready to use.
Interesting insight already: Most Nashville businesses cluster at 4.0 stars (1,493). The distribution leans positive, as customers tend to rate places they liked. This is called positivity bias in review data. 

### Step 5: Filter Down to Coffee and Cafe Businesses
This filters the full business table down to only coffee related businesses using a LOWER and LIKE pattern match on the categories column, then summarizes average stars, review totals, and open versus closed counts. This becomes the core filtered dataset used for the rest of the analysis.


In [0]:
-- =============================================
-- STEP 5: Count coffee/cafe businesses
-- =============================================
SELECT
  COUNT(*) AS total_coffee_businesses,
  ROUND(AVG(stars), 2) AS avg_stars,
  MIN(stars) AS min_stars,
  MAX(stars) AS max_stars,
  SUM(review_count) AS total_reviews,
  SUM(CASE WHEN is_open = 1 THEN 1 ELSE 0 END) AS currently_open,
  SUM(CASE WHEN is_open = 0 THEN 1 ELSE 0 END) AS permanently_closed
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%';

total_coffee_businesses,avg_stars,min_stars,max_stars,total_reviews,currently_open,permanently_closed
394,3.7,1.0,5.0,37881,266,128


**Result:** About a third of Nashville coffee shops have closed, showing this is a competitive market where location and quality matter. The average rating sits around 3.7, which means there is real room for a new shop to stand out by aiming for 4.0 stars or higher.

Two big insights for CoffeeKing:

33% of Nashville coffee shops closed, the market is competitive. Location and quality matter a lot.
Avg rating 3.7, means there is room for CoffeeKing to stand out by targeting 4.0+ quality.

### Step 6: Identify the Most Reviewed Coffee Shops
This ranks coffee shops by total review count and returns the top 10, along with their star rating, zip code, and open or closed status. High review counts usually indicate established, high traffic locations, which are useful reference points for competitive analysis.

In [0]:
-- =============================================
-- STEP 6: Top 10 coffee shops by review count
-- =============================================
SELECT
  name,
  stars,
  review_count,
  postal_code,
  is_open,
  CASE WHEN is_open = 1 THEN 'Open' ELSE 'Closed' END AS status
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
ORDER BY review_count DESC
LIMIT 10;

name,stars,review_count,postal_code,is_open,status
Milk and Honey Nashville,4.0,1725,37203,1,Open
Fido,4.0,1260,37212,1,Open
Crema Coffee Roasters,4.5,998,37210,1,Open
Frothy Monkey,4.0,988,37219,1,Open
Frothy Monkey,4.0,852,37204,1,Open
Sky Blue Cafe,4.0,836,37206,1,Open
Another Broken Egg Cafe,4.0,818,37201,1,Open
The Garden Brunch Cafe,4.5,712,37208,0,Closed
Germantown Cafe,4.0,662,37208,1,Open
Barista Parlor,3.5,600,37206,1,Open


### Step 7: Star Rating Distribution Among Coffee Shops
This groups coffee shops by star rating and calculates what percentage of all coffee shops each rating represents, using a window function so percentages are calculated across the whole filtered group.




In [0]:
-- =============================================
-- STEP 7: Star rating distribution (coffee only)
-- =============================================
SELECT
  stars,
  COUNT(*) AS shop_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
GROUP BY stars
ORDER BY stars;

stars,shop_count,percentage
1.0,3,0.8
1.5,14,3.6
2.0,26,6.6
2.5,23,5.8
3.0,34,8.6
3.5,58,14.7
4.0,110,27.9
4.5,100,25.4
5.0,26,6.6


Databricks visualization. Run in Databricks to view.

**Result:** About 53% of Nashville coffee shops are rated 4.0 stars or higher, which is above the overall Nashville business average. This suggests coffee customers are more demanding, but they reward shops that meet their expectations generously. It also supports the idea that top performing shops cluster near 4.0 rather than spreading toward the extremes.

Big insight: 53.3% of Nashville coffee shops are rated 4.0 or above. This is higher than the overall Nashville business average. Coffee customers are harder to satisfy but reward good shops generously.
This directly proves Hypothesis 2, high review shops cluster at 4.0, not at extremes. 

### Step 8: Question 1, Which Zip Codes Have the Best Coffee Shops
This groups coffee shops by postal code and calculates average stars, total reviews, and open versus closed counts for each zip code, keeping only zip codes with at least 3 shops so the averages are meaningful, then sorts by rating.



In [0]:
-- =============================================
-- STEP 8: Q1 - Best zip codes for coffee shops
-- =============================================
SELECT
  CAST(postal_code AS STRING)        AS zip_code,
  COUNT(*)                           AS total_shops,
  ROUND(AVG(stars), 2)               AS avg_stars,
  SUM(review_count)                  AS total_reviews,
  SUM(CASE WHEN is_open = 1 
      THEN 1 ELSE 0 END)             AS open_shops,
  SUM(CASE WHEN is_open = 0 
      THEN 1 ELSE 0 END)             AS closed_shops
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
GROUP BY postal_code
HAVING COUNT(*) >= 3
ORDER BY avg_stars DESC, total_reviews DESC;

zip_code,total_shops,avg_stars,total_reviews,open_shops,closed_shops
37220,3,4.67,93,2,1
37206,31,4.24,4125,19,12
37216,13,4.19,626,9,4
37208,19,4.16,3102,10,9
37204,22,3.77,3502,16,6
37201,15,3.77,3052,10,5
37209,28,3.77,2250,22,6
37219,21,3.76,1961,12,9
37210,11,3.68,1869,8,3
37212,21,3.67,3200,12,9


**Result:** Zip code 37203 has the most coffee shops in Nashville at 78 locations, but only a 3.65 average rating, suggesting an overcrowded market with lower average quality. Zip code 37206 stands out as the strongest opportunity, with 31 shops, a 4.24 average rating, and 4,125 total reviews, showing strong demand paired with strong quality. Zip code 37220 has the highest rating at 4.67 but only 3 shops and 93 reviews, meaning the market there is too small to draw firm conclusions yet.

Three huge insights for CoffeeKing:

37203 has 78 coffee shops (most in Nashville) but only 3.65 avg stars, this confirms Hypothesis 1. 
Downtown = overcrowded + lower quality perception.
37206 is the sweet spot with 31 shops, 4.24 stars, 4,125 reviews, 19 still open. High demand, high quality neighbourhood. Best candidate for CoffeeKing location.
37220 has highest rating (4.67) but only 3 shops and 93 reviews which is too small as a market, but worth watching.


### Step 9: Question 2, Does Review Volume Affect Star Rating
This buckets coffee shops into four groups based on review count (low, mid, high, very high) and compares the average star rating within each bucket, to test whether shops with more reviews tend to score differently than shops with fewer reviews.



In [0]:
-- =============================================
-- STEP 9: Q2 - Review volume vs star rating
-- =============================================
SELECT
  CASE
    WHEN review_count < 50   THEN '1. Low (under 50)'
    WHEN review_count < 200  THEN '2. Mid (50-199)'
    WHEN review_count < 500  THEN '3. High (200-499)'
    ELSE                          '4. Very High (500+)'
  END                            AS review_bucket,
  COUNT(*)                       AS shop_count,
  ROUND(AVG(stars), 2)           AS avg_stars,
  MIN(stars)                     AS min_stars,
  MAX(stars)                     AS max_stars,
  ROUND(AVG(review_count), 0)    AS avg_reviews_in_bucket
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
GROUP BY review_bucket
ORDER BY review_bucket;

review_bucket,shop_count,avg_stars,min_stars,max_stars,avg_reviews_in_bucket
1. Low (under 50),231,3.53,1.0,5.0,23.0
2. Mid (50-199),116,3.89,1.0,5.0,98.0
3. High (200-499),34,4.04,3.5,5.0,299.0
4. Very High (500+),13,4.12,3.5,4.5,859.0


**Result:** Every bucket shows a higher average rating as review volume increases, with the busiest shops averaging 4.12 stars. This disproves the original hypothesis that high review shops would simply cluster around 3.5 to 4.0. Instead, popularity and quality move together: shops that satisfy customers attract more visits, which generates more reviews, which reinforces the higher rating. The takeaway is that focusing on quality first tends to bring reviews and ratings up together.

The pattern is undeniable as more reviews = higher rating. Every bucket goes up. No regression to mean.

This DISPROVES Hypothesis 2 as high-review shops would cluster at 3.5-4.0. Actually they score HIGHER at 4.12. Real data beat the hypothesis. Hypothesis disproved, finding was opposite.

Why does this happen? Popular shops get more customers → more reviews → customers who love a place leave more reviews than those who had average experience. Quality drives both popularity and ratings together.
CoffeeKing takeaway: Focus on quality first → reviews and ratings will follow together.

### Step 10: Question 3, Do Opening Hours Affect Star Rating
This classifies each coffee shop into a schedule type (7 days a week, Monday through Saturday, weekdays only, or irregular) by checking the hours field for specific day entries, then compares average star rating and total reviews across each schedule type.




In [0]:
-- =============================================
-- STEP 10: Q3 - Opening hours vs star rating
-- =============================================
SELECT
  CASE
    WHEN hours LIKE '%"Monday":"%' 
     AND hours LIKE '%"Saturday":"%'
     AND hours LIKE '%"Sunday":"%'   THEN '7 days a week'
    WHEN hours LIKE '%"Monday":"%'
     AND hours LIKE '%"Saturday":"%' THEN 'Mon-Sat (6 days)'
    WHEN hours LIKE '%"Monday":"%'   THEN 'Weekdays only'
    ELSE 'Other/Irregular'
  END                               AS schedule_type,
  COUNT(*)                          AS shop_count,
  ROUND(AVG(stars), 2)              AS avg_stars,
  SUM(review_count)                 AS total_reviews,
  SUM(CASE WHEN is_open = 1 
      THEN 1 ELSE 0 END)            AS currently_open
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
GROUP BY schedule_type
ORDER BY avg_stars DESC;

schedule_type,shop_count,avg_stars,total_reviews,currently_open
Mon-Sat (6 days),37,4.26,3305,21
Weekdays only,22,4.05,1169,10
Other/Irregular,56,3.84,3418,24
7 days a week,279,3.57,29989,211


**Result:** Shops open 7 days a week actually score lower on average (around 3.57) than shops open Monday through Saturday (around 4.26). This disproves the hypothesis that being open every day leads to better ratings. A likely explanation is that 7 day operations tend to be larger chain style cafes with less consistent quality, while Monday through Saturday shops tend to be smaller, more focused independent shops. The strategic takeaway is that consistent quality across fewer days can outperform being open every single day.

This COMPLETELY DISPROVES Hypothesis 3. 7-day shops would have better ratings. The opposite is true.
Why does this make sense when you think about it:

7-day shops = mostly chain cafes, high volume, inconsistent quality.
Mon-Sat shops = smaller independent specialty coffee shops, more focused, higher quality.
Weekdays only = office-area artisan cafes serving professionals who expect quality.

CoffeeKing strategic recommendation forming:
Don't open 7 days just to seem available. Open Mon-Sat with exceptional quality → data says that scores 4.26 avg stars vs 3.57 for 7-day shops. Quality over availability.

### Step 11: Final Summary, Best Target Zip Codes for CoffeeKing
This combines everything learned so far into one summary table: total shops, average stars, total and average reviews, open versus closed counts, and a closure rate percentage per zip code, filtered to zip codes with at least 3 shops and sorted to surface the best rated, lowest churn areas first. This is meant to be the primary reference table for choosing a location.

In [0]:
-- ===============================================================================
-- STEP 11: Final summary - Top target zip codes for CoffeeKing with full context
-- ===============================================================================
SELECT
  CAST(postal_code AS STRING)          AS zip_code,
  COUNT(*)                             AS total_shops,
  ROUND(AVG(stars), 2)                 AS avg_stars,
  SUM(review_count)                    AS total_reviews,
  ROUND(AVG(review_count), 0)          AS avg_reviews_per_shop,
  SUM(CASE WHEN is_open = 1 
      THEN 1 ELSE 0 END)               AS open_shops,
  SUM(CASE WHEN is_open = 0 
      THEN 1 ELSE 0 END)               AS closed_shops,
  ROUND(SUM(CASE WHEN is_open = 0 
      THEN 1 ELSE 0 END) * 100.0 / 
      COUNT(*), 1)                     AS closure_rate_pct
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
GROUP BY postal_code
HAVING COUNT(*) >= 3
ORDER BY avg_stars DESC, closure_rate_pct ASC
LIMIT 8;

zip_code,total_shops,avg_stars,total_reviews,avg_reviews_per_shop,open_shops,closed_shops,closure_rate_pct
37220,3,4.67,93,31.0,2,1,33.3
37206,31,4.24,4125,133.0,19,12,38.7
37216,13,4.19,626,48.0,9,4,30.8
37208,19,4.16,3102,163.0,10,9,47.4
37209,28,3.77,2250,80.0,22,6,21.4
37204,22,3.77,3502,159.0,16,6,27.3
37201,15,3.77,3052,203.0,10,5,33.3
37219,21,3.76,1961,93.0,12,9,42.9


## Milestone 2: Descriptive Statistics

### Full Descriptive Stats for Coffee Businesses
This calculates the core descriptive statistics for the coffee shop dataset in one query: mean, standard deviation, minimum, maximum, and the 25th, 50th, and 75th percentiles for both stars and review_count. These numbers describe the shape and spread of the data, which is a standard first step before drawing statistical conclusions.

In [0]:
-- =============================================
-- MILESTONE 2: DESCRIPTIVE STATISTICS
-- =============================================

-- CELL 12: Full descriptive stats for coffee businesses
SELECT
  COUNT(*)                            AS total_shops,
  ROUND(AVG(stars), 3)                AS mean_stars,
  ROUND(STDDEV(stars), 3)             AS stddev_stars,
  MIN(stars)                          AS min_stars,
  MAX(stars)                          AS max_stars,
  ROUND(PERCENTILE(stars, 0.25), 1)   AS q1_stars,
  ROUND(PERCENTILE(stars, 0.50), 1)   AS median_stars,
  ROUND(PERCENTILE(stars, 0.75), 1)   AS q3_stars,
  ROUND(AVG(review_count), 1)         AS mean_reviews,
  ROUND(STDDEV(review_count), 1)      AS stddev_reviews,
  MIN(review_count)                   AS min_reviews,
  MAX(review_count)                   AS max_reviews,
  ROUND(PERCENTILE(review_count, 0.50), 0) AS median_reviews
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%';

total_shops,mean_stars,stddev_stars,min_stars,max_stars,q1_stars,median_stars,q3_stars,mean_reviews,stddev_reviews,min_reviews,max_reviews,median_reviews
394,3.702,0.923,1.0,5.0,3.0,4.0,4.5,96.1,174.4,5,1725,39.0


### Review Count Patterns Across Star Rating Tiers
This groups coffee shops into four star rating tiers (top tier, good, average, and poor) and, within each tier, calculates the average, standard deviation, minimum, maximum, median review count, and number of currently open shops. This shows whether higher rated shops also tend to attract more reviews and stay open more often.


In [0]:
-- ============================================================================
-- CELL 13: Stars vs review count correlation bucket analysis with IQR context
-- ============================================================================
SELECT
  CASE
    WHEN stars >= 4.5 THEN '4.5 to 5.0 (top tier)'
    WHEN stars >= 4.0 THEN '4.0 to 4.4 (good)'
    WHEN stars >= 3.0 THEN '3.0 to 3.9 (average)'
    ELSE                   'below 3.0 (poor)'
  END                               AS star_tier,
  COUNT(*)                          AS shop_count,
  ROUND(AVG(review_count),1)        AS avg_reviews,
  ROUND(STDDEV(review_count),1)     AS stddev_reviews,
  MIN(review_count)                 AS min_reviews,
  MAX(review_count)                 AS max_reviews,
  ROUND(PERCENTILE(review_count,0.5),0) AS median_reviews,
  SUM(CASE WHEN is_open=1 THEN 1 ELSE 0 END) AS open_shops
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
GROUP BY star_tier
ORDER BY avg_reviews DESC;

star_tier,shop_count,avg_reviews,stddev_reviews,min_reviews,max_reviews,median_reviews,open_shops
4.0 to 4.4 (good),110,146.5,263.8,5,1725,56.0,64
4.5 to 5.0 (top tier),126,99.3,146.2,5,998,42.0,86
3.0 to 3.9 (average),92,78.7,108.6,5,600,33.0,61
below 3.0 (poor),66,30.5,20.5,5,85,30.0,55


### Zip Code Quality Compared to Shop Survival Rate
This groups coffee shops by zip code, restricted to zip codes with at least 5 shops, and calculates average and median stars, average and median reviews, open and closed counts, and a survival rate percentage. Sorting by survival rate first shows which neighborhoods not only rate well but also keep their coffee shops open long term.


In [0]:
-- ===============================================================================
-- CELL 14: Milestone 2 - Cross analysis zip code quality vs open/closed survival
-- ===============================================================================
SELECT
  CAST(postal_code AS STRING)            AS zip_code,
  COUNT(*)                               AS total_shops,
  ROUND(AVG(stars),2)                    AS avg_stars,
  ROUND(STDDEV(stars),2)                 AS stddev_stars,
  ROUND(PERCENTILE(stars,0.5),1)         AS median_stars,
  ROUND(AVG(review_count),0)             AS avg_reviews,
  ROUND(PERCENTILE(review_count,0.5),0)  AS median_reviews,
  SUM(CASE WHEN is_open=1 THEN 1 ELSE 0 END)  AS open_shops,
  SUM(CASE WHEN is_open=0 THEN 1 ELSE 0 END)  AS closed_shops,
  ROUND(SUM(CASE WHEN is_open=1 THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS survival_rate_pct
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
GROUP BY postal_code
HAVING COUNT(*) >= 5
ORDER BY survival_rate_pct DESC, avg_stars DESC;

zip_code,total_shops,avg_stars,stddev_stars,median_stars,avg_reviews,median_reviews,open_shops,closed_shops,survival_rate_pct
37207,8,3.25,1.34,4.0,42.0,33.0,7,1,87.5
37217,8,3.0,1.36,2.8,43.0,45.0,7,1,87.5
37215,15,3.63,0.83,4.0,81.0,49.0,12,3,80.0
37209,28,3.77,1.04,4.0,80.0,34.0,22,6,78.6
37204,22,3.77,0.74,4.0,159.0,78.0,16,6,72.7
37210,11,3.68,1.12,4.0,170.0,17.0,8,3,72.7
37203,78,3.65,0.73,3.8,90.0,38.0,56,22,71.8
37214,30,3.28,1.1,3.3,68.0,33.0,21,9,70.0
37216,13,4.19,0.9,4.5,48.0,36.0,9,4,69.2
37201,15,3.77,0.98,4.0,203.0,39.0,10,5,66.7


### Correlation Between Stars, Reviews, and Open Status
This calculates Pearson style correlation coefficients between star rating and review count, star rating and open status, and review count and open status. A correlation close to 0 means little to no linear relationship, while a value closer to 1 or negative 1 means a strong relationship. This adds statistical backing to the patterns already noticed in earlier steps.

In [0]:
-- ========================================================================
-- CELL 15: Pearson-style correlation stars vs review_count (coffee shops)
-- ========================================================================
SELECT
  ROUND(CORR(stars, review_count), 4)        AS corr_stars_reviews,
  ROUND(CORR(stars, is_open), 4)             AS corr_stars_isopen,
  ROUND(CORR(review_count, is_open), 4)      AS corr_reviews_isopen,
  COUNT(*)                                   AS total_shops
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%';

corr_stars_reviews,corr_stars_isopen,corr_reviews_isopen,total_shops
0.1587,-0.1186,0.1435,394


### New Metric: Quality Score
This introduces a custom metric, Quality Score, calculated as stars multiplied by the base 10 logarithm of review_count plus 1. The logarithm keeps very high review counts from completely dominating the score, so this metric rewards shops that combine a strong rating with meaningful review volume, rather than just picking whichever shop has the single highest rating or highest review count.

In [0]:
-- =============================================
-- CELL 16: NEW METRIC 1
-- Quality Score = stars * LOG(review_count + 1)
-- Rewards both high ratings AND volume
-- =============================================
SELECT
  name,
  stars,
  review_count,
  CAST(postal_code AS STRING)                          AS zip_code,
  is_open,
  ROUND(stars * LOG10(review_count + 1), 3)            AS quality_score,
  CASE WHEN is_open = 1 THEN 'Open' ELSE 'Closed' END AS status
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
ORDER BY quality_score DESC
LIMIT 15;

name,stars,review_count,zip_code,is_open,quality_score,status
Crema Coffee Roasters,4.5,998,37210,1,13.498,Open
Milk and Honey Nashville,4.0,1725,37203,1,12.948,Open
The Garden Brunch Cafe,4.5,712,37208,0,12.839,Closed
Mike's Ice Cream,4.5,593,37201,1,12.482,Open
Fido,4.0,1260,37212,1,12.403,Open
Sunflower Cafe,4.5,561,37204,1,12.374,Open
Frothy Monkey,4.0,988,37219,1,11.981,Open
VN Pho & Deli,4.5,440,37209,1,11.9,Open
Frothy Monkey,4.0,852,37204,1,11.724,Open
Sky Blue Cafe,4.0,836,37206,1,11.691,Open


### New Metric: Market Opportunity Index
This introduces a second custom metric, Opportunity Index, built per zip code from three weighted parts: average star rating (50 percent weight), a lower shop count relative to 100 (30 percent weight, rewarding less saturated markets), and survival rate (20 percent weight). Zip codes with at least 3 shops are ranked highest opportunity first. This metric is designed to directly answer where CoffeeKing should consider opening, balancing quality, competition level, and business survival together in one number.

In [0]:
-- =================================================================
-- CELL 17: NEW METRIC 2
-- Market Opportunity Index per zip code
-- High avg stars + low shop count + low closure = best opportunity
-- =================================================================
SELECT
  CAST(postal_code AS STRING)                          AS zip_code,
  COUNT(*)                                             AS total_shops,
  ROUND(AVG(stars), 2)                                 AS avg_stars,
  ROUND(SUM(CASE WHEN is_open=1 THEN 1.0 ELSE 0 END)
        / COUNT(*) * 100, 1)                           AS survival_pct,
  ROUND(AVG(review_count), 0)                          AS avg_reviews,
  ROUND(
    (AVG(stars) / 5.0) * 0.5 +
    (1.0 - COUNT(*) / 100.0) * 0.3 +
    (SUM(CASE WHEN is_open=1 THEN 1.0 ELSE 0 END)
     / COUNT(*)) * 0.2
  , 4)                                                 AS opportunity_index
FROM business
WHERE LOWER(categories) LIKE '%coffee%'
   OR LOWER(categories) LIKE '%cafe%'
   OR LOWER(categories) LIKE '%cafes%'
   OR LOWER(categories) LIKE '%coffee roasters%'
GROUP BY postal_code
HAVING COUNT(*) >= 3
ORDER BY opportunity_index DESC
LIMIT 10;

zip_code,total_shops,avg_stars,survival_pct,avg_reviews,opportunity_index
37220,3,4.67,66.7,31.0,0.891
37216,13,4.19,69.2,48.0,0.8187
37210,11,3.68,72.7,170.0,0.7806
37215,15,3.63,80.0,81.0,0.7783
37207,8,3.25,87.5,42.0,0.776
37201,15,3.77,66.7,203.0,0.765
37208,19,4.16,52.6,163.0,0.7641
37204,22,3.77,72.7,159.0,0.7567
37206,31,4.24,61.3,133.0,0.7538
37217,8,3.0,87.5,43.0,0.751


### Going Broader: Comparing Business Ratings to Individual Review Ratings
This joins the business table to the review table to compare each shop's overall published star rating against the actual average of its individual review scores, keeping only shops with at least 50 reviews for reliability. The star_gap column shows the difference between the two. A large gap can mean a shop's public rating is out of date, or that its rating has shifted recently due to a run of very positive or very negative reviews.


In [0]:
-- =============================================
-- CELL 18: GO BROADER
-- Join business + review tables
-- Find avg review-level stars vs business stars
-- Are business ratings consistent with reviews?
-- =============================================
SELECT
  b.name,
  b.stars                                    AS business_stars,
  ROUND(AVG(r.stars), 2)                     AS avg_review_stars,
  ROUND(b.stars - AVG(r.stars), 2)           AS star_gap,
  COUNT(r.review_id)                         AS review_count,
  CAST(b.postal_code AS STRING)              AS zip_code
FROM business b
JOIN review r
  ON b.business_id = r.business_id
WHERE LOWER(b.categories) LIKE '%coffee%'
   OR LOWER(b.categories) LIKE '%cafe%'
   OR LOWER(b.categories) LIKE '%cafes%'
   OR LOWER(b.categories) LIKE '%coffee roasters%'
GROUP BY b.name, b.stars, b.postal_code
HAVING COUNT(r.review_id) >= 50
ORDER BY ABS(b.stars - AVG(r.stars)) DESC
LIMIT 15;

name,business_stars,avg_review_stars,star_gap,review_count,zip_code
212 Cafe and Wellness Center,4.0,3.71,0.29,102,37203
Starbucks,3.0,2.74,0.26,96,37211
Amy's Ethiopian Restaurant,4.5,4.24,0.26,99,37217
Stay Golden,4.0,4.25,-0.25,176,37204
Bongo Java,4.0,3.75,0.25,104,37203
Gojo Ethiopian,4.5,4.25,0.25,229,37211
Listening Room Cafe,4.0,3.75,0.25,565,37210
Dose Café & Dram Bar,4.5,4.26,0.24,133,37216
Little Gourmand,5.0,4.76,0.24,74,37215
Johnny Cash Museum,4.0,4.24,-0.24,420,37201
